# 2D weld pool (scikit-fem): heat, flow, solidification

Version 1 for aluminium arc welding of a **monolithic plate** (joint gap not resolved).

**This model (FEM):**
- 2D longitudinal section \(x\) (along weld), \(z\) (thickness; mesh coordinate `y`)
- Arc → moving **Gaussian surface heat flux**
- Taylor–Hood \(P_2\)–\(P_1\) velocity–pressure, \(P_1\) temperature (scikit-fem)
- Liquid / solid via linear \(f_L\) + Darcy–Carman–Kozeny sink in the mush
- Driving forces: **Marangoni** (natural BC) + buoyancy (Boussinesq)
- Default momentum: unsteady **Stokes** + Darcy (stable on coarse grids); set `enable_inertia=True` for \((\mathbf{u}\cdot\nabla)\mathbf{u}\) + SUPG
- Energy advection with SUPG; **no** EM, arc pressure, free-surface deformation, or filler

**References** (`refs/weld_pool`):
- He, Fuerschbach, DebRoy, *J. Phys. D* 36 (2003) — transient heat/flow, Gaussian flux, Carman–Kozeny
- Zhao & DebRoy, *Metall. Mater. Trans. B* 32B (2001) — Al 5182 properties, negative \(d\sigma/dT\)
- Pitscheneder et al., *Welding Journal* (1996) — convection / Peclet controls pool shape
- Kidess et al., *Int. J. Thermal Sci.* 104 (2016) — laminar viscosity under-mixes (no effective-viscosity tuning)

## Weak form (summary)

Momentum (backward Euler + Picard on advection):

$$
\frac{\rho}{\Delta t}(\mathbf{u}-\mathbf{u}^n,\mathbf{v})
+ 2\mu\,(\varepsilon(\mathbf{u}),\varepsilon(\mathbf{v}))
+ \bigl(A(f_L)\mathbf{u},\mathbf{v}\bigr)
+ \rho\bigl((\mathbf{u}^*\cdot\nabla)\mathbf{u},\mathbf{v}\bigr)_{\mathrm{SUPG}}
- (p,\nabla\cdot\mathbf{v})
+ (q,\nabla\cdot\mathbf{u})
= \bigl(\rho g\beta(T-T_\mathrm{ref})\,\mathbf{e}_z,\mathbf{v}\bigr)
+ \int_\mathrm{top} f_L\frac{d\sigma}{dT}\partial_x T\, v_x\,\mathrm{d}s
$$

with \(A(f_L)=C(1-f_L)^2/(f_L^3+B)\).
On the default coarse grid the convective (SUPG) block is off
(`enable_inertia=False` → unsteady Stokes + Darcy).

Energy with apparent heat capacity in the mush \(T_s<T<T_l\):

$$
\rho c_{p,\mathrm{eff}}\partial_t T + \rho c_p\,\mathbf{u}\cdot\nabla T
= \nabla\cdot(k\nabla T),
\qquad
c_{p,\mathrm{eff}}=c_p+\frac{L}{T_l-T_s}\ \text{(in mush)}
$$

Top flux: Gaussian in − convection − radiation. Walls/bottom: \(T=T_0\), no-slip (top: \(w=0\) + Marangoni).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name != 'heat+flow_fem_model':
    cand = ROOT / 'heat+flow_fem_model'
    if cand.is_dir():
        sys.path.insert(0, str(cand))
        MODEL_DIR = cand
    else:
        sys.path.insert(0, str(ROOT))
        MODEL_DIR = ROOT
else:
    sys.path.insert(0, str(ROOT))
    MODEL_DIR = ROOT

from weldpool_fem import Material, Process, WeldPoolFEM, quick_process

print('model dir:', MODEL_DIR)
print('scikit-fem solver: WeldPoolFEM')

## Material (Al 5182) and process parameters

Defaults from Zhao & DebRoy (2001), Table III.  
`QUICK = True` — coarse short run (~tens of seconds). Set `False` for the finer default grid (minutes).

In [ ]:
QUICK = False  # set False for finer Process() defaults

mat = Material()  # Al 5182

if QUICK:
    proc = quick_process(
        enable_flow=True,
        t_end=0.35,
        q_max=8.0e7,   # W/m^2 peak (effective 2D)
        r_b=0.0025,
        v_weld=0.008,
        x0=0.006,
    )
else:
    proc = Process(
        enable_flow=True,
        t_end=2.0,
        q_max=6.0e7,
        r_b=0.003,
        v_weld=0.005,
        x0=0.008,
        nx=40,
        nz=12,
        dt=5e-3,
        picard_iter=2,
    )

print('grid', proc.nx, 'x', proc.nz, '  Lx,Lz [mm] =', 1e3*proc.Lx, 1e3*proc.Lz)
print('q_max =', proc.q_max, 'W/m^2   rb =', proc.r_b, 'm   v =', proc.v_weld, 'm/s')
print('dt =', proc.dt, 's   picard =', proc.picard_iter)
print('Ts,Tl =', mat.T_solidus, mat.T_liquidus, 'K   dsigma/dT =', mat.dsigma_dT)

## Run with flow

In [ ]:
model = WeldPoolFEM(material=mat, process=proc)
diag = model.run(record_every=0.05, progress=True)

print('heat balance |Qin - Qloss - Qdir - dH| / Qin =', round(model.heat_balance_ratio(), 4))
print('final Tmax =', round(model.T.max(), 1), 'K')
depth, length = model.pool_metrics()
print('pool depth, length [mm] =', round(1e3*depth, 2), round(1e3*length, 2))
print('max |u| =', round(max(diag.max_speed), 4), 'm/s   max Pe =', round(max(diag.pe_number), 2))

## Fields: temperature, liquid fraction, velocity

In [ ]:
def plot_fields(m, title_prefix=''):
    T, fL, u_c, w_c = m.sample_fields()
    Xmm = 1e3 * m.X
    Zmm = 1e3 * m.Z
    fig, axs = plt.subplots(1, 3, figsize=(14, 3.6), constrained_layout=True)

    im0 = axs[0].pcolormesh(Xmm, Zmm, T, shading='auto', cmap='inferno')
    axs[0].contour(Xmm, Zmm, T, levels=[m.mat.T_solidus, m.mat.T_liquidus],
                   colors=['cyan', 'lime'], linewidths=1.2)
    axs[0].axvline(1e3 * m.arc_centre(), color='w', ls='--', lw=0.8)
    fig.colorbar(im0, ax=axs[0], label='T [K]')
    axs[0].set_title(f'{title_prefix}Temperature')

    im1 = axs[1].pcolormesh(Xmm, Zmm, fL, shading='auto', cmap='Blues', vmin=0, vmax=1)
    fig.colorbar(im1, ax=axs[1], label='f_L')
    axs[1].set_title('Liquid fraction')

    speed = np.sqrt(u_c**2 + w_c**2)
    im2 = axs[2].pcolormesh(Xmm, Zmm, speed, shading='auto', cmap='viridis')
    sx, sz = max(1, m.proc.nx // 20), max(1, m.proc.nz // 6)
    axs[2].quiver(
        Xmm[::sx, ::sz], Zmm[::sx, ::sz],
        u_c[::sx, ::sz], w_c[::sx, ::sz],
        color='w', scale=None, width=0.003,
    )
    fig.colorbar(im2, ax=axs[2], label='|V| [m/s]')
    axs[2].set_title('Velocity')

    for ax in axs:
        ax.set_xlabel('x [mm]')
        ax.set_ylabel('z [mm]')
        ax.set_aspect('equal')
    return fig

plot_fields(model, title_prefix=f't={model.t:.2f}s  ')
plt.show()

## Pool size, thermal cycle, fusion zone

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.5), constrained_layout=True)
t = np.array(diag.time)

axs[0].plot(t, 1e3 * np.array(diag.pool_depth), label='depth')
axs[0].plot(t, 1e3 * np.array(diag.pool_length), label='length')
axs[0].set_xlabel('t [s]'); axs[0].set_ylabel('mm'); axs[0].legend(); axs[0].set_title('Pool size')
axs[0].grid(True, alpha=0.3)

axs[1].plot(t, diag.T_probe, color='C1')
axs[1].axhline(mat.T_solidus, color='c', ls='--', label='Ts')
axs[1].axhline(mat.T_liquidus, color='g', ls='--', label='Tl')
axs[1].set_xlabel('t [s]'); axs[1].set_ylabel('T [K]'); axs[1].legend()
axs[1].set_title(f'Thermocycle (probe dof {model.probe_dof})'); axs[1].grid(True, alpha=0.3)

T_max_grid = model._eval_T_at(np.vstack([model.X.ravel(), model.Y.ravel()])).reshape(model.X.shape)
# use stored nodal T_max projected to grid
from scipy.interpolate import LinearNDInterpolator
interp = LinearNDInterpolator(model.basis_T.doflocs.T, model.T_max)
Tmax_g = np.nan_to_num(interp(np.column_stack([model.X.ravel(), model.Y.ravel()])), nan=proc.T0).reshape(model.X.shape)
fz = Tmax_g >= mat.T_liquidus
im = axs[2].pcolormesh(1e3*model.X, 1e3*model.Z, Tmax_g, shading='auto', cmap='hot')
axs[2].contour(1e3*model.X, 1e3*model.Z, fz.astype(float), levels=[0.5], colors='cyan')
fig.colorbar(im, ax=axs[2], label='T_max [K]')
axs[2].set_title('Fusion zone (T_max > Tl)')
axs[2].set_xlabel('x [mm]'); axs[2].set_ylabel('z [mm]'); axs[2].set_aspect('equal')
plt.show()

## Checks

1. Heat balance: \(|Q_\mathrm{in}-Q_\mathrm{loss}-Q_\mathrm{Dirichlet}-\Delta H|/Q_\mathrm{in}\)
2. Velocity ≈ 0 in solid (\(f_L\approx 0\))
3. Marangoni: for \(d\sigma/dT<0\), surface outflow from the hot spot
4. Solidification after the arc passes (\(f_L\to 0\) behind the torch)
5. Optional: conduction-only vs with flow

In [ ]:
T, fL, u_c, w_c = model.sample_fields()
speed = np.sqrt(u_c**2 + w_c**2)
solid = fL < 1e-6
liquid = fL > 0.5

print('=== Heat balance ===')
print('ratio =', round(model.heat_balance_ratio(), 4))

print('=== Solid velocity ===')
print('max |V| in solid =', float(speed[solid].max()) if np.any(solid) else 0.0)
print('max |V| in liquid =', float(speed[liquid].max()) if np.any(liquid) else 0.0)

print('=== Marangoni direction (top surface) ===')
u_left, u_right = model.marangoni_surface_means()
corr = model.marangoni_outflow_corr()
corr_hist = np.array(diag.marangoni_corr, dtype=float)
corr_peak = np.nanmax(corr_hist) if np.any(np.isfinite(corr_hist)) else float('nan')
print('mean u left of arc =', u_left, ' (expect < 0)')
print('mean u right of arc =', u_right, ' (expect > 0)')
print('outflow corr (final) =', corr, '  peak during run =', corr_peak, ' (expect > 0)')
ok_m = (mat.dsigma_dT < 0) and (corr_peak > 0 or (u_left < 0 and u_right > 0))
print('Marangoni check:', 'OK' if ok_m else 'weak / check mid-melt snapshot')

print('=== Solidification behind arc ===')
xc = model.arc_centre()
behind = model.X < xc - 2*proc.r_b
if np.any(behind):
    print('max f_L behind arc (x < xarc-2 rb) =', float(fL[behind].max()))
else:
    print('domain still under / ahead of torch — extend t_end to see freeze-back')

In [ ]:
# Conduction-only comparison (same process, short)
t_cmp = min(0.25, proc.t_end)
proc_cond = quick_process(
    enable_flow=False,
    t_end=t_cmp,
    q_max=proc.q_max,
    r_b=proc.r_b,
    v_weld=proc.v_weld,
    x0=proc.x0,
    nx=proc.nx,
    nz=proc.nz,
    Lx=proc.Lx,
    Lz=proc.Lz,
    dt=proc.dt,
    dt_max=proc.dt_max,
)
m_cond = WeldPoolFEM(material=mat, process=proc_cond)
m_cond.run(record_every=0.1, progress=False)

proc_flow = quick_process(
    enable_flow=True,
    t_end=t_cmp,
    q_max=proc.q_max,
    r_b=proc.r_b,
    v_weld=proc.v_weld,
    x0=proc.x0,
    nx=proc.nx,
    nz=proc.nz,
    Lx=proc.Lx,
    Lz=proc.Lz,
    dt=proc.dt,
    dt_max=proc.dt_max,
    picard_iter=proc.picard_iter,
)
m_flow = WeldPoolFEM(material=mat, process=proc_flow)
m_flow.run(record_every=0.1, progress=False)
d_c, L_c = m_cond.pool_metrics()
d_f, L_f = m_flow.pool_metrics()
print(f'At t={t_cmp}s:')
print(f'  conduction-only: depth={1e3*d_c:.2f} mm  length={1e3*L_c:.2f} mm  Tmax={m_cond.T.max():.0f} K')
print(f'  with flow:       depth={1e3*d_f:.2f} mm  length={1e3*L_f:.2f} mm  Tmax={m_flow.T.max():.0f} K')
print(f'  Pe (flow) = {m_flow.peclet():.1f}')
print(f'  Marangoni corr (flow) = {m_flow.marangoni_outflow_corr():.4f}')

## Notes / next versions

- Filler wire / droplet heat and mass not included (monolithic plate).
- No Lorentz force — add MHD in a later version.
- Flat free surface; no arc pressure.
- Molecular viscosity only (Kidess: underestimates mixing).
- 2D omits lateral heat loss: calibrate \(q_\max, r_b\) as effective parameters.
- Finite-difference companion: `heat+flow_2d/`.